In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

catalog_name = 'ecommerce'

## Brands

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_brands')
display(df_bronze.limit(10))

In [0]:
df_silver = df_bronze.withColumn("brand_name",trim(col("brand_name")))
display(df_silver.limit(10))

In [0]:
df_silver = df_silver.withColumn("brand_code",regexp_replace(col("brand_code"),r'[^A-Za-z0-9]',''))

display(df_silver.limit(10))

In [0]:
df_silver.select("category_code").distinct().display()

In [0]:
anomalies = {
    "GROCERY":"GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}

df_silver = df_silver.replace(to_replace=anomalies,subset=["category_code"])

df_silver.select("category_code").distinct().display()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

## Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
display(df_bronze.limit(10))

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter(col("count") > 1)
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(['category_code'])
display(df_silver)

In [0]:
df_silver = df_silver.withColumn("category_code",upper(col("category_code")))
display(df_silver)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

## Products

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

row_count , column_count = df_bronze.count(),len(df_bronze.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
display(df_bronze.limit(5))

In [0]:
df_bronze.select("weight_grams").show(5,truncate=False)

In [0]:
df_silver = df_bronze.withColumn("weight_grams",regexp_replace(col("weight_grams"),"g","").cast(IntegerType()))

df_silver.select("weight_grams").show(5,truncate=False)

In [0]:
df_silver.select("length_cm").show(3)

In [0]:
df_silver = df_silver.withColumn(
    "length_cm",
    regexp_replace(col("length_cm"),",",".").cast(FloatType())
)

df_silver.select("length_cm").show(3)

In [0]:
df_silver.select("category_code","brand_code").show(2)

In [0]:
df_silver = df_silver.withColumn(
    "category_code",
    upper(col("category_code"))
).withColumn(
    "brand_code",
    upper(col("brand_code"))
)

df_silver.select("category_code","brand_code").show(2)

In [0]:
df_silver.select("material").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "material",
    when(col("material") == "Coton","Cotton")
    .when(col("material") == "Alumium","Aluminum")
    .when(col("material") == "Ruber","Rubber")
    .otherwise(col("material"))
)

df_silver.select("material").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "rating_count",
    when(col("rating_count").isNotNull(),abs(col("rating_count"))).otherwise(lit(0))
)

In [0]:
df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10,truncate=False)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

## Customers

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

row_count , column_count = df_bronze.count(),len(df_bronze.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

display(df_bronze.limit(10))

In [0]:
null_count = df_bronze.filter(col("customer_id").isNull()).count()

print(f"Null count: {null_count}")

In [0]:
df_bronze.filter(col("customer_id").isNull()).show(3)

In [0]:
df_silver = df_bronze.dropna(subset=["customer_id"])

row_count = df_silver.count()
print(f"Row count after dropping null values: {row_count}")

In [0]:
null_count = df_silver.filter(col("phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}")

In [0]:
df_silver.filter(col("phone").isNull()).show(3)

In [0]:
df_silver = df_silver.fillna("Not Available",subset=["phone"])

df_silver.filter(col("phone").isNull()).show()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

## Calendar

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

row_count , column_count = df_bronze.count(),len(df_bronze.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

In [0]:
df_bronze.printSchema()

In [0]:
df_silver = df_bronze.withColumn("date",to_date(df_bronze["date"],"dd-MM-yyyy"))

print(df_silver.printSchema())

df_silver.show(5)

In [0]:
duplicates = df_silver.groupBy('date').count().filter("count > 1")

print("Total duplicated row: ",duplicates.count())
display(duplicates)

In [0]:
df_silver = df_silver.dropDuplicates(["date"])

row_count = df_silver.count()

print("Rows After removing duplicates: ",row_count)

In [0]:
df_silver = df_silver.withColumn("day_name", initcap(col("day_name")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn("week_of_year",abs(col("week_of_year")))

df_silver.show(3)

In [0]:
df_silver = df_silver.withColumn("quarter",concat_ws("",concat(lit("Q"),col("quarter"),lit("-"),col("year"))))

df_silver = df_silver.withColumn("week_of_year",concat_ws("-",concat(lit("Week"),col("week_of_year"),lit("-"),col("year"))))

df_silver.show(3)

In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year","week")

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")